Dataset Preparation

In [3]:
import argparse
import os
import numpy as np
from typing import Tuple

Сore geometry and routing functions

In [6]:
def dist(a,b):
  #Eucledean distance between two 2D points.
  return float(np.sqrt((a[0] - b[0])**2 + (a[1] - b[1])**2))

In [9]:
def nn_route_length(depot, pts):
  #Nearest-neighbor routing length
  #depot -> all points -> depot
  if len(pts) == 0:
    return 0.0

  unvisited = pts.copy()
  cur = depot
  total_len = 0.0

  while unvisited:
    idx = min(
        range(len(unvisited)),
        key=lambda i: dist(cur, unvisited[i])
    )
    nxt = unvisited.pop(idx)
    total_len += dist(cur, nxt)
    cur = nxt

  total_len += dist(cur, depot)
  return total_len

In [16]:
#Test NN-routing on a known example
depot = (0, 0)
pts = [(1,0), (0,1), (1,1)]

nn_route_length(depot, pts)

4.0

Brute-force optimal assignment

In [12]:
def best_assignment_two_courriers(depot, points_xy):
  """ Enumerate all 2^n assigments and finds the one minimizing
  the makespan under NN routing.
  Returns: best_makespan, best_mask, route_len_A, route_len_B"""

  N = len(points_xy)
  best = None

  for mask in range(1 << N):
    A = [points_xy[i] for i in range(N) if ((mask >> i) & 1) == 0]
    B = [points_xy[i] for i in range(N) if ((mask >> i) & 1) == 1]

    LA = nn_route_length(depot, A)
    LB = nn_route_length(depot, B)
    makespan = max(LA, LB)

    if best is None or makespan < best[0]:
      best = (makespan, mask, LA, LB)

  return best


In [17]:
pts = [(1,0), (0,1), (1,1), (2,0)]
best_assignment_two_courriers((0,0), pts)

(4.0, 6, 4.0, 3.414213562373095)

Dataset Generation 

In [13]:
#Dataset generation
def make_dataset(num_samples, N, seed, low=0.0, high=2.0):
  """ Generates (X, y) dataset.
  X shape: (num_samples, 2N)
  y shape: (num_samples, N)"""
  rng = np.random.default_rng(seed)
  depot = (0.0, 0.0)

  X = np.zeros((num_samples, 2 * N), dtype = np.float32)
  y = np.zeros((num_samples, N), dtype=np.int64)

  for s in range(num_samples):
    pts = rng.uniform(low=low, high=high, size=(N,2)).astype(np.float32)

    best_makespan, best_mask, _, _ = best_assignment_two_courriers(
        depot, pts.tolist()
    )

    X[s] = pts.reshape(-1)
    for i in range(N):
      y[s, i] = (best_mask >> i) & 1

  return X, y

In [20]:
#generate a tiny dataset for sanity check
""" This cell verifies that the dataset generator
produces correctly shaped inputs and optimal binary
courier assignments, and that the assignments are geometrically
consistent with makespan minimization."""
X, y = make_dataset(num_samples=5, N=4, seed=42)
X,y

(array([[1.5479121 , 0.8777569 , 1.7171959 , 1.394736  , 0.1883547 ,
         1.9512447 , 1.5222794 , 1.5721287 ],
        [0.25622725, 0.90077186, 0.74159604, 1.8535299 , 1.2877302 ,
         1.6455232 , 0.8868284 , 0.45447743],
        [1.1091696 , 0.12763451, 1.6552624 , 1.2633288 , 1.5161755 ,
         0.7090519 , 1.941396  , 1.7862422 ],
        [1.556767  , 0.38927743, 0.933442  , 0.08760753, 0.308579  ,
         1.3660979 , 1.4895244 , 1.9350195 ],
        [0.6516507 , 0.7409194 , 0.93911165, 0.37894273, 0.25984302,
         0.9514099 , 0.4538187 , 1.339628  ]], dtype=float32),
 array([[0, 0, 1, 0],
        [1, 1, 0, 0],
        [1, 1, 1, 0],
        [1, 1, 1, 0],
        [1, 1, 0, 0]]))

Splitting the dataset

In [14]:
#Train/Valid/ Test split

def split_dataset(X, y, seed, ratios=(0.8, 0.1, 0.1)):
  #Splits dataset into train/val/test

  assert abs(sum(ratios) - 1.0) < 1e-6

  rng = np.random.default_rng(seed)
  idx = rng.permutation(len(X))

  n_train = int(ratios[0] * len(X))
  n_val = int(ratios[1] * len(X))

  train_idx = idx[:n_train]
  val_idx = idx[n_train:n_train + n_val]
  test_idx = idx[n_train + n_val:]

  return (
      X[train_idx], y[train_idx],
      X[val_idx], y[val_idx],
      X[test_idx], y[test_idx]
  )

In [22]:
#split check
X_train, y_train, X_val, y_val, X_test, y_test = split_dataset(X,y, seed=42)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(4, 8) (4, 4)
(0, 8) (0, 4)
(1, 8) (1, 4)


Dataset generation Final 

In [15]:
#Save dataset to .npz

def save_npz(path, X_train, y_train, X_val, y_val, X_test, y_test, meta):
  #Saves dataset splits and metadata to .npz

  np.savez (
      path,
      X_train=X_train,
      y_train=y_train,
      X_val=X_val,
      y_val=y_val,
      X_test=X_test,
      y_test=y_test,
      meta=meta
  )

In [ ]:
#Full dataset generation (final)
X, y = make_dataset(num_samples=5000, N=8, seed=0)

X_train, y_train, X_val, y_val, X_test, y_test = split_dataset(X, y, seed=0)

save_npz("./dataset",
         X_train, y_train,
         X_val, y_val,
         X_test, y_test,
         meta= {
             "N": 8,
             "seed": 0,
             "num_samples": 5000,
             "coord_low": 0.0,
             "coord_high": 2.0
         })